In [17]:
import json
import pandas as pd
import asyncio
import os
from dotenv import load_dotenv
import google.generativeai as genai
from tqdm.auto import tqdm

load_dotenv('../.env')
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

data = json.load(open('../data/anchored_labeled.json'))
df = pd.DataFrame(data)
df['anchor_dt'] = pd.to_datetime(df['anchor_dt'])

print(f'전체 청크: {len(df)}개')
print(df['llm_label'].value_counts())

전체 청크: 385개
llm_label
실습    188
개념     83
예시     72
기타     42
Name: count, dtype: int64


In [18]:
# 기타 제외, 파일별 시간순 정렬
seq_df = (
    df[df['llm_label'] != '기타']
    .sort_values(['file', 'anchor_dt'])
    .reset_index(drop=True)
)

# 연속 쌍 생성 (같은 파일 내)
seq_df['next_label'] = seq_df.groupby('file')['llm_label'].shift(-1)
seq_df['next_dt']    = seq_df.groupby('file')['anchor_dt'].shift(-1)
seq_df['next_text']  = seq_df.groupby('file')['text'].shift(-1)

# 파일 마지막 행 제거 (next가 없는 쌍)
pairs = seq_df.dropna(subset=['next_label']).copy()
print(f'연속 쌍 수: {len(pairs)}개')
print()
print('레이블 쌍 분포:')
print(pairs.groupby(['llm_label', 'next_label']).size().unstack(fill_value=0))

연속 쌍 수: 328개

레이블 쌍 분포:
next_label  개념   실습  예시
llm_label              
개념          27   35  20
실습          41  109  25
예시          14   36  21


In [20]:
GEMINI_MODEL = 'gemini-2.5-flash'

def make_skip_prompt(concept_text: str, key_sentence: str, next_text: str, next_label: str) -> str:
    return f"""당신은 백엔드 입문 강의를 평가하는 교육 전문가입니다.

아래는 강의 전사 텍스트에서 개념 설명으로 분류된 구간과, 그 바로 다음에 이어지는 구간입니다.

[개념 설명 — 핵심 정의 문장]
{key_sentence}

[개념 설명 — 전체 맥락]
{concept_text[:1200]}

[다음 구간 ({next_label})]
{str(next_text)[:600]}

---

이 개념 설명 다음에 별도의 예시 없이 바로 {next_label}으로 넘어갔습니다.
이 흐름에서 **예시가 없어도 학습자가 개념을 이해할 수 있는지** 판단하세요.
개념 자체의 성질과 이어지는 {next_label} 내용을 함께 고려합니다.

**예시가 필요한 개념 (needs_example: true)**
- 동작 원리·메커니즘 개념이라 "이게 실제로 어떻게 작동하는지"를 보여줘야 이해됨
- 여러 요소가 얽힌 구조라서 전체 흐름을 한 번 보여줘야 감이 잡히는 개념
- 정의만으로는 "언제, 어떤 상황에서 쓰는지"가 불분명하고, 이어지는 {next_label}에서도 그 맥락이 채워지지 않음

**예시가 필요하지 않은 개념 (needs_example: false)**
- 단순 명칭·용어 소개로 정의 자체가 직관적임
- 앞서 배운 개념의 소폭 확장이거나 맥락상 자연스럽게 따라오는 개념
- 정의 자체에 사용 방법이 내포되어 있어 추가 예시 없이도 적용 가능한 개념
- 이어지는 {next_label} 내용을 통해 이 개념이 어떻게 쓰이는지 자연스럽게 드러남

애매한 경우 false로 판단하세요.

아래 JSON 형식으로만 응답하세요:
{{
  "needs_example": true 또는 false,
  "reason": "판단 근거 — 개념의 성질 또는 이어지는 흐름 중 어느 쪽 기준인지 한 줄로"
}}"""

async def classify_skip_single(model, semaphore, pbar, idx, row):
    async with semaphore:
        try:
            response = await asyncio.to_thread(
                model.generate_content,
                make_skip_prompt(
                    row['text'],
                    row.get('llm_key_sentence') or '',
                    row.get('next_text') or '',
                    row.get('next_label') or '다음 구간',
                ),
                generation_config=genai.GenerationConfig(response_mime_type='application/json')
            )
            result = json.loads(response.text)
        except Exception as e:
            result = {'needs_example': None, 'reason': str(e)}
        finally:
            pbar.update(1)
        return {'idx': idx, **result}

async def run_skip_classification(df_input, concurrency=15):
    model = genai.GenerativeModel(GEMINI_MODEL)
    semaphore = asyncio.Semaphore(concurrency)
    pbar = tqdm(total=len(df_input), desc='예시 필요 여부 판단 중')
    tasks = [
        classify_skip_single(model, semaphore, pbar, idx, row)
        for idx, row in df_input.iterrows()
    ]
    results = await asyncio.gather(*tasks)
    pbar.close()
    return results

print('함수 정의 완료')

함수 정의 완료


## 순서 위반 탐지

키워드 겹침(overlap coefficient) 기준으로 동일 주제를 판별하고, 주제 덩어리 내 순서 위반을 탐지.

- **위반 A**: 동일 주제 내 실습이 예시보다 먼저 등장
- **위반 B**: 동일 주제 내 예시가 없을 때 LLM이 예시 필요 여부 판단 → 필요한 경우만 위반


In [45]:
import re
from kiwipiepy import Kiwi

kiwi = Kiwi()

STOPWORDS = {
    '다음', '경우', '질문', '설명', '내용', '부분', '얘기', '이야기',
    '생각', '문제', '방법', '기본', '처음', '마지막', '단계', '순서',
    '사람', '분들', '여러분', '선생님', '강사', '학생', '구현', '키워드',
}

def extract_keywords(text):
    tokens = kiwi.tokenize(text[:500])
    return set(
        t.form for t in tokens
        if t.tag in ('NNG', 'NNP', 'SL')
        and len(t.form) >= 2
        and t.form not in STOPWORDS
    )

def overlap_coef(kw_a, kw_b):
    if not kw_a or not kw_b:
        return 0.0
    return len(kw_a & kw_b) / min(len(kw_a), len(kw_b))

seq_df_kw = seq_df.copy().reset_index(drop=True)
seq_df_kw['keywords'] = seq_df_kw['text'].apply(extract_keywords)
print(f'키워드 추출 완료: {len(seq_df_kw)}개')
print(f'샘플: {list(seq_df_kw["keywords"].iloc[0])[:10]}')

키워드 추출 완료: 343개
샘플: ['옵션', '사방', '스타일', '생성', '나머지', '정리', '일반', '컨텐츠', '이남', '리턴']


In [53]:
KW_THRESHOLD = 0.1  # 조정 가능

def detect_violations_by_keyword(seq_df_kw, threshold=KW_THRESHOLD):
    viol_a_rows = []
    viol_b_cands = []
    topic_groups = []

    for file, grp in seq_df_kw.groupby('file'):
        grp = grp.sort_values('anchor_dt').reset_index(drop=True)
        concept_idxs = grp.index[grp['llm_label'] == '개념'].tolist()

        for k, ci in enumerate(concept_idxs):
            concept_row = grp.iloc[ci]
            concept_kw  = concept_row['keywords']
            end_idx = concept_idxs[k + 1] if k + 1 < len(concept_idxs) else len(grp)
            window  = grp.iloc[ci + 1:end_idx]
            prac_ex = window[window['llm_label'].isin(['실습', '예시'])].copy()

            base = {
                'file': file,
                'dt':   concept_row['anchor_dt'],
                'text': concept_row['text'],
                'llm_key_sentence': concept_row.get('llm_key_sentence', ''),
            }

            if len(prac_ex) == 0:
                topic_groups.append({**base, 'pattern': '개념만', 'n_예시': 0, 'n_실습': 0})
                viol_b_cands.append({**base, 'seq_str': '개념만', 'next_text': '', 'next_label': '다음 구간'})
                continue

            prac_ex = prac_ex.copy()
            prac_ex['overlap'] = prac_ex['keywords'].apply(lambda kw: overlap_coef(concept_kw, kw))
            same_topic = prac_ex[prac_ex['overlap'] >= threshold].sort_values('anchor_dt')

            examples  = same_topic[same_topic['llm_label'] == '예시']
            practices = same_topic[same_topic['llm_label'] == '실습']
            n_ex, n_prac = len(examples), len(practices)

            if n_ex > 0 and n_prac > 0:
                pat = '개념+예시+실습'
            elif n_ex > 0:
                pat = '개념+예시'
            elif n_prac > 0:
                pat = '개념+실습'
            else:
                pat = '개념만'
            topic_groups.append({**base, 'pattern': pat, 'n_예시': n_ex, 'n_실습': n_prac})

            seq_str = ' → '.join(same_topic['llm_label'].tolist()) if len(same_topic) > 0 else '동일주제없음'

            if n_ex == 0:
                first_prac = practices.iloc[0] if n_prac > 0 else window.iloc[0]
                viol_b_cands.append({
                    **base,
                    'seq_str':    seq_str,
                    'next_text':  first_prac['text'],
                    'next_label': first_prac['llm_label'],
                })
            elif n_prac > 0:
                first_ex_dt = examples['anchor_dt'].min()
                if len(practices[practices['anchor_dt'] < first_ex_dt]) > 0:
                    viol_a_rows.append({**base, 'seq_str': seq_str})

    return pd.DataFrame(viol_a_rows), pd.DataFrame(viol_b_cands), pd.DataFrame(topic_groups)


viol_a_kw_df, viol_b_kw_cands_df, topic_kw_df = detect_violations_by_keyword(seq_df_kw, KW_THRESHOLD)

print(f'=== 키워드 기반 주제 덩어리 분포 (임계값={KW_THRESHOLD}) ===')
print(topic_kw_df['pattern'].value_counts().to_string())
print()
print(f'위반 A (동일주제 내 실습 먼저): {len(viol_a_kw_df)}건')
print(f'위반 B 후보 (동일주제 내 예시 없음): {len(viol_b_kw_cands_df)}건')

=== 키워드 기반 주제 덩어리 분포 (임계값=0.1) ===
pattern
개념만         39
개념+실습       22
개념+예시+실습    13
개념+예시        9

위반 A (동일주제 내 실습 먼저): 4건
위반 B 후보 (동일주제 내 예시 없음): 61건


In [56]:
# 위반 B 후보 LLM 판단
viol_b_kw_llm_results = await run_skip_classification(viol_b_kw_cands_df, concurrency=15)

llm_kw_df = pd.DataFrame(viol_b_kw_llm_results).set_index('idx')
viol_b_kw_cands_df['needs_example'] = llm_kw_df['needs_example']
viol_b_kw_cands_df['skip_reason']   = llm_kw_df['reason']

viol_b_kw_df = viol_b_kw_cands_df[viol_b_kw_cands_df['needs_example'] == True].copy()

print('=== 위반 B LLM 판단 결과 ===')
print(viol_b_kw_cands_df['needs_example'].value_counts())
print()
print(f'위반 A: {len(viol_a_kw_df)}건')
print(f'위반 B: {len(viol_b_kw_df)}건')
print(f'전체:   {len(viol_a_kw_df) + len(viol_b_kw_df)}건')

예시 필요 여부 판단 중: 100%|██████████| 61/61 [00:50<00:00,  1.22it/s]

=== 위반 B LLM 판단 결과 ===
needs_example
True     52
False     9
Name: count, dtype: int64

위반 A: 4건
위반 B: 52건
전체:   56건


In [ ]:
# 파일별 최종 요약
def per_file_counts(df, col_name):
    if len(df) == 0:
        return pd.Series(dtype=int).rename(col_name)
    return df.groupby('file').size().rename(col_name)

summary_kw = (
    pd.concat([
        per_file_counts(viol_a_kw_df, '위반A'),
        per_file_counts(viol_b_kw_df, '위반B'),
    ], axis=1)
    .fillna(0).astype(int)
)
summary_kw['합계'] = summary_kw['위반A'] + summary_kw['위반B']
summary_kw = summary_kw.sort_values('합계', ascending=False)

print(f'=== 파일별 위반 건수 (키워드 임계값={KW_THRESHOLD}) ===')
print(summary_kw.to_string())
print()
print(f'위반 A (동일주제 실습→예시 순서 위반): {len(viol_a_kw_df)}건')
print(f'위반 B (예시 누락 + LLM 필요 판단):   {len(viol_b_kw_df)}건')
print(f'전체 위반: {len(viol_a_kw_df) + len(viol_b_kw_df)}건')
print()
print('=== 개념 블록 패턴 분포 ===')
print(topic_kw_df['pattern'].value_counts().to_string())
